# Inteligentna prognoza cen mieszkań w Polsce

Interaktywny projekt Data Science dla Google Colab. Notebook pokazuje mapę powiatów, prognozuje cenę mieszkania dla wybranego metrażu i ocenia model regresji liniowej.

**Ważne:** notebook próbuje pobrać publiczne dane kontekstowe z GUS BDL i NBP. GUS BDL nie udostępnia w prostym, stabilnym endpointcie kompletnej tabeli cen mieszkań dla każdego powiatu, dlatego ceny za m² są generowane jako dane demonstracyjne. Zostanie to wyraźnie pokazane w wynikach.

In [ ]:
# Instalacja bibliotek — uruchom tę komórkę jako pierwszą.
!pip -q install numpy pandas scikit-learn plotly ipywidgets requests

In [ ]:
import hashlib
import json
import warnings
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
np.random.seed(42)

# Colab potrzebuje tego ustawienia, aby FigureWidget działał interaktywnie.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print('Środowisko:', 'Google Colab' if IN_COLAB else 'Jupyter / lokalne')

## 1. Pobranie granic powiatów i danych kontekstowych

Granice powiatów są pobierane jako GeoJSON. Kod próbuje także odczytać listę jednostek powiatowych z GUS BDL oraz ostatnią cenę złota z API NBP. Jeżeli źródło jest niedostępne, notebook nie przerywa pracy — wykorzystuje dane awaryjne i pokazuje komunikat.

In [ ]:
GEOJSON_URL = 'https://raw.githubusercontent.com/jusuff/PolandGeoJson/main/data/poland.counties.json'
GUS_UNITS_URL = 'https://bdl.stat.gov.pl/api/v1/units?level=5&page-size=500&format=json'
NBP_GOLD_URL = 'https://api.nbp.pl/api/cenyzlota/last/1/?format=json'

def get_json(url, timeout=30):
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()

geojson = get_json(GEOJSON_URL)
features = geojson.get('features', [])

# Wspólny identyfikator mapy: kod TERC z właściwości GeoJSON.
map_units = pd.DataFrame([
    {'terc': str(f.get('properties', {}).get('terc', '')),
     'powiat': f.get('properties', {}).get('name', 'Nieznany')}
    for f in features
])
map_units = map_units[map_units['terc'].ne('')].drop_duplicates('terc').reset_index(drop=True)

gus_status = 'niedostępne — użyto nazw z GeoJSON'
try:
    gus_json = get_json(GUS_UNITS_URL, timeout=15)
    gus_rows = gus_json.get('results', gus_json.get('data', []))
    gus_units = pd.DataFrame(gus_rows)
    if not gus_units.empty:
        # Dane GUS potwierdzają listę jednostek, ale nie zastępują cen mieszkań.
        gus_status = f'dostępne — {len(gus_units)} jednostek odczytanych z GUS BDL'
except Exception as error:
    gus_units = pd.DataFrame()
    gus_status = f'niedostępne ({type(error).__name__})'

nbp_gold = np.nan
nbp_status = 'niedostępne'
try:
    nbp_json = get_json(NBP_GOLD_URL, timeout=15)
    nbp_gold = float(nbp_json[0]['cena'])
    nbp_status = f'dostępne — cena złota: {nbp_gold:.2f} zł/g'
except Exception as error:
    nbp_status = f'niedostępne ({type(error).__name__})'

display(Markdown(f'''
**Status źródeł:**  
- Granice powiatów: pobrane z publicznego GeoJSON.
- GUS BDL: {gus_status}.
- NBP: {nbp_status}.
- **Ceny mieszkań za m²: dane symulowane demonstracyjnie.**
'''))
print('Liczba powiatów na mapie:', len(map_units))

## 2. Przygotowanie danych demonstracyjnych

Ceny są symulowane, ale nie są losowe bez zależności biznesowych. Wyższy indeks urbanizacji podnosi cenę, a największe miasta otrzymują premię. Stałe ziarno generatora zapewnia powtarzalność wyników.

In [ ]:
def stable_number(text):
    return int(hashlib.md5(str(text).encode('utf-8')).hexdigest()[:8], 16)

rng = np.random.default_rng(42)
data = map_units.copy()

# Powiaty grodzkie i największe miasta dostają realistyczną premię cenową.
major_city_bonus = {
    'Warszawa': 9000, 'Kraków': 5000, 'Wrocław': 3500, 'Gdańsk': 3500,
    'Poznań': 2800, 'Gdynia': 2600, 'Sopot': 4200, 'Katowice': 1500,
    'Łódź': 1000, 'Szczecin': 1200, 'Lublin': 700, 'Rzeszów': 500
}
data['indeks_urbanizacji'] = data['powiat'].map(lambda x: 0.25 + (stable_number(x) % 76) / 100)
data['premia_miasta'] = data['powiat'].map(lambda x: next((v for k, v in major_city_bonus.items() if k.lower() in str(x).lower()), 0))
gold_adjustment = 0 if pd.isna(nbp_gold) else (nbp_gold - 450) * 0.35
data['cena_m2'] = (
    5200
    + data['indeks_urbanizacji'] * 3100
    + data['premia_miasta']
    + gold_adjustment
    + rng.normal(0, 350, len(data))
).clip(3500, 30000).round(0)

# Kilka obserwacji dla każdego powiatu umożliwia trening modelu.
n_samples_per_county = 6
model_data = data.loc[data.index.repeat(n_samples_per_county)].reset_index(drop=True)
model_data['metraz'] = rng.integers(30, 121, len(model_data))
model_data['cena_m2_obserwowana'] = (model_data['cena_m2'] + rng.normal(0, 220, len(model_data))).clip(3000)
model_data['cena_calkowita'] = (model_data['metraz'] * model_data['cena_m2_obserwowana'] + rng.normal(0, 18000, len(model_data))).clip(50000).round(0)

print(f'Powiaty: {len(data)} | Obserwacje treningowe: {len(model_data)}')
display(data[['terc', 'powiat', 'indeks_urbanizacji', 'cena_m2']].head())

## 3. Model regresji liniowej i ocena jakości

Model prognozuje całkowitą cenę mieszkania na podstawie metrażu oraz cech powiatu. Cena za m² jest następnie wyliczana jako prognozowana cena całkowita podzielona przez metraż.

In [ ]:
features = ['metraz', 'cena_m2', 'indeks_urbanizacji', 'premia_miasta']
X = model_data[features]
y = model_data['cena_calkowita']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)

metrics = pd.DataFrame({
    'Metryka': ['R²', 'MAE'],
    'Wartość': [round(r2, 3), round(mae, 0)],
    'Interpretacja': [
        'Im bliżej 1, tym lepsze dopasowanie modelu.',
        'Średni błąd prognozy w złotych.'
    ]
})
display(metrics)

comparison = pd.DataFrame({'Rzeczywista cena': y_test, 'Prognozowana cena': predictions})
fig_quality = px.scatter(comparison, x='Rzeczywista cena', y='Prognozowana cena',
                         title='Ocena modelu: wartości rzeczywiste i prognozowane')
fig_quality.show()

## 4. Wykres porównujący ceny w powiatach

Poniżej znajduje się ranking 20 powiatów o najwyższej symulowanej cenie za m².

In [ ]:
top20 = data.nlargest(20, 'cena_m2').sort_values('cena_m2')
fig_bar = px.bar(top20, x='cena_m2', y='powiat', orientation='h',
                 title='Porównanie cen mieszkań w powiatach — top 20',
                 labels={'cena_m2': 'Cena za m² (zł)', 'powiat': 'Powiat'},
                 color='cena_m2', color_continuous_scale='Viridis')
fig_bar.show()

## 5. Gotowa aplikacja interaktywna

Kliknij powiat na mapie i przesuń suwak metrażu. Aplikacja pokaże prognozowaną cenę za m² oraz całkowitą cenę mieszkania.

In [ ]:
# Interaktywna mapa Plotly. Kliknięcie obszaru zmienia wybrany powiat.
map_fig = px.choropleth(
    data,
    geojson=geojson,
    locations='terc',
    featureidkey='properties.terc',
    color='cena_m2',
    color_continuous_scale='YlOrRd',
    hover_name='powiat',
    hover_data={'terc': False, 'cena_m2': ':,.0f', 'indeks_urbanizacji': ':.2f'},
    labels={'cena_m2': 'Cena za m²', 'indeks_urbanizacji': 'Urbanizacja'},
    title='Ceny mieszkań według powiatu — dane symulowane'
)
map_fig.update_geos(fitbounds='locations', visible=False)
map_fig.update_layout(height=650, margin={'r': 0, 't': 60, 'l': 0, 'b': 0})
map_widget = go.FigureWidget(map_fig)

selected = {'terc': data.iloc[0]['terc']}
area_slider = widgets.IntSlider(value=60, min=30, max=120, step=1, description='Metraż:', continuous_update=True, style={'description_width': 'initial'})
info_output = widgets.Output()

def selected_row():
    row = data.loc[data['terc'].astype(str).eq(str(selected['terc']))]
    return row.iloc[0] if not row.empty else data.iloc[0]

def show_prediction(*args):
    row = selected_row()
    area = area_slider.value
    input_row = pd.DataFrame([{
        'metraz': area,
        'cena_m2': row['cena_m2'],
        'indeks_urbanizacji': row['indeks_urbanizacji'],
        'premia_miasta': row['premia_miasta']
    }])
    total_price = max(0, float(model.predict(input_row)[0]))
    price_m2 = total_price / area
    with info_output:
        clear_output(wait=True)
        display(Markdown(f'''
### Wynik prognozy
**Wybrany powiat:** {row['powiat']}  
**Metraż:** {area} m²  
**Prognozowana cena za m²:** {price_m2:,.0f} zł  
**Prognozowana cena całkowita:** {total_price:,.0f} zł  

*Wynik ma charakter demonstracyjny, ponieważ ceny wejściowe są symulowane.*
'''))

def on_map_click(trace, points, selector):
    if points.point_inds:
        point_index = points.point_inds[0]
        try:
            selected['terc'] = str(trace.locations[point_index])
        except Exception:
            selected['terc'] = str(data.iloc[point_index]['terc'])
        show_prediction()

map_widget.data[0].on_click(on_map_click)
area_slider.observe(show_prediction, names='value')
show_prediction()

display(Markdown('## Aplikacja'))
display(widgets.VBox([area_slider, info_output]))
display(map_widget)
display(Markdown('**Instrukcja:** kliknij dowolny powiat na mapie. Zmiana suwaka metrażu natychmiast aktualizuje prognozę.'))

## Interpretacja i ograniczenia

Model regresji liniowej pokazuje zależność między metrażem, poziomem urbanizacji, premią dużego miasta i ceną całkowitą. R² opisuje część zmienności wyjaśnianą przez model, a MAE podaje przeciętny błąd w złotych.

Ceny mieszkań w tym notebooku są **symulowane** i służą celom dydaktycznym. Nie należy wykorzystywać ich do wyceny nieruchomości ani decyzji inwestycyjnych. Model nie uwzględnia standardu lokalu, piętra, roku budowy, dzielnicy, lokalizacji względem transportu, podaży, popytu ani bieżących transakcji.